# Publication copy
The code and analytical commentary are preserved. Stored execution outputs were cleared to avoid publishing respondent previews or local execution details. Vetted aggregate tables and figures are provided in `course_artifacts`; see the repository results guide. The fully executed originals remain in the local course workspace. This notebook needs the separately acquired survey data and protected fit caches for a complete replay. No models were refitted for publication.


# Approximate refinement among the six retained solutions

We compare saved complete-response assignments; no clustering is refitted and no held-out respondents are consulted. A finer partition refines a coarser partition when every finer cluster lies inside one coarse cluster. Independent K-means partitions need not be nested. We therefore seek an approximate refinement graph, not assume a mathematical lattice.

All overlaps and purities use equal-country weights recalculated on the complete cohort. Standard ARI remains unweighted. A–F identify solutions, not a ranking.

## 1. Recover the common population and assignments

**What and why:** reconstruct exactly the complete-response cohort used in v3, verify saved fingerprints and assignment lengths, and calculate equal-country weights. Every pair must refer to the same respondents in the same order. The reference input and design fingerprint are checked before comparing labels.

In [ ]:
from pathlib import Path
import json,hashlib,itertools
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score
from IPython.display import display,Markdown
ROOT=Path.cwd()
if not (ROOT/'course_artifacts').exists(): ROOT=ROOT.parent
SOURCE=ROOT/'course_artifacts/week04_experiment1_complete_overall_worry_v3'
OUTPUT=SOURCE/'refinement'; OUTPUT.mkdir(exist_ok=True)
INPUT=ROOT/'course_artifacts/week04_search_v4/stage1_eligible_development.parquet'
design=json.loads((SOURCE/'design.json').read_text())
fingerprint=hashlib.sha256(INPUT.read_bytes()+json.dumps(design,sort_keys=True).encode()).hexdigest()
domains=['food','water','crime','severe_weather','traffic','mental_health']
worry=['worry_'+d for d in domains]
harm=['harm_'+d+'_'+kind for kind in ['personal','known_person'] for d in domains+['work']]
raw=pd.read_parquet(INPUT)
eligible=raw[worry].notna().sum(axis=1).ge(4)&raw[harm[:7]].notna().sum(axis=1).ge(4)
data=raw.loc[eligible&raw[worry+harm].notna().all(axis=1)].reset_index(drop=True)
assert data.country_split.eq('train').all() and data.WPID_RANDOM.is_unique
weights=(data.WGT/data.groupby('COUNTRY_ISO3').WGT.transform('sum')/data.COUNTRY_ISO3.nunique()).to_numpy()
solutions={'A':('012','worry_emphasized',3),'B':('012','balanced',4),
           'C':('013','worry_emphasized',6),'D':('013','worry_emphasized',7),
           'E':('013','worry_emphasized',8),'F':('015','worry_emphasized',6)}
labels={}
for name,(encoding,balance,k) in solutions.items():
    saved=joblib.load(SOURCE/f'reference_{encoding}_{balance}_k{k}.joblib')
    assert saved['fingerprint']==fingerprint and len(saved['labels'])==len(data)
    labels[name]=saved['labels']
    assert np.array_equal(np.unique(labels[name]),np.arange(k))
display(pd.DataFrame([(name,*spec) for name,spec in solutions.items()],columns=['ID','encoding','balance','k']))
print('Common respondents:',len(data),'countries:',data.COUNTRY_ISO3.nunique())


## 2. Calculate all pairwise relationships

**What this cell does:** builds weighted overlap tables, using coarse clusters as rows and finer clusters as columns. Refinement purity R is the sum of each column's largest overlap. Individual column purity is that overlap divided by the finer cluster's share. Exact refinement has R=1.

**Why include a baseline:** if assignments were independent, mapping every fine cluster to the largest coarse group would achieve its share b. We report (R−b)/(1−b) as dominance-baseline-adjusted purity. This is not an exact finite-sample chance correction. The worst column purity identifies a small group hidden by a high overall score.

For equal k (C versus F), many-to-one mapping is insufficient: we additionally report optimal one-to-one weighted agreement using Hungarian matching, and both directional purities. All comparisons are descriptive; no sampling uncertainty is estimated here.

In [ ]:
rows=[]; overlaps=[]; cluster_rows=[]
matrix=pd.DataFrame(np.nan,index=solutions,columns=solutions)
for name in solutions: matrix.loc[name,name]=1.
for first,second in itertools.combinations(solutions,2):
    if solutions[first][2]>solutions[second][2]: first,second=second,first
    k1,k2=solutions[first][2],solutions[second][2]
    table=np.bincount(labels[first].astype(int)*k2+labels[second],weights=weights,minlength=k1*k2).reshape(k1,k2)
    column_share=table.sum(axis=0)
    purity=table.max(axis=0)/column_share
    score=table.max(axis=0).sum()
    baseline=table.sum(axis=1).max()
    reverse=table.max(axis=1).sum()
    r,c=linear_sum_assignment(-table)
    agreement=table[r,c].sum() if k1==k2 else np.nan
    rows.append({'coarse':first,'fine':second,'coarse_k':k1,'fine_k':k2,'refinement_purity':score,
                 'worst_fine_purity':purity.min(),'dominance_baseline':baseline,
                 'baseline_adjusted_purity':(score-baseline)/(1-baseline),
                 'reverse_purity':reverse,'one_to_one_agreement':agreement,
                 'unweighted_ARI':adjusted_rand_score(labels[first],labels[second])})
    matrix.loc[first,second]=score
    if k1==k2: matrix.loc[second,first]=reverse
    for j in range(k2):
        cluster_rows.append({'coarse':first,'fine':second,'fine_cluster':j,
                             'fine_share':column_share[j],'best_coarse_cluster':int(table[:,j].argmax()),'purity':purity[j]})
    for i in range(k1):
        for j in range(k2):
            overlaps.append({'coarse':first,'fine':second,'coarse_cluster':i,'fine_cluster':j,
                             'joint_share':table[i,j],'pct_of_coarse':100*table[i,j]/table[i].sum(),
                             'pct_of_fine':100*table[i,j]/column_share[j]})
results=pd.DataFrame(rows)
clusters=pd.DataFrame(cluster_rows)
overlap=pd.DataFrame(overlaps)
for name,t in [('pairwise_refinement',results),('fine_cluster_purity',clusters),('membership_overlaps',overlap)]:
    t.to_csv(OUTPUT/f'{name}.csv',index=False)
matrix.to_csv(OUTPUT/'refinement_matrix.csv')
display(Markdown('### All 15 pairwise comparisons'))
display(results.round(4))
display(Markdown('### Refinement matrix: row = coarse solution; column = finer solution'))
display(matrix.round(3))


## 3. Inspect overlaps and graph eligibility

**What and why:** plot all 15 weighted overlap tables as percentages of each finer cluster. A single dark cell in each column supports nesting. Several substantial cells show boundary reorganization. Separate mapping of distinct fine groups into one coarse group is allowed.

For a readable illustrative graph, flag only different-k relationships with R≥0.90 and every finer-cluster purity≥0.80. These are provisional visualization thresholds, not statistical significance or validated scientific cutoffs. We do not infer transitive edges or perform a transitive reduction: every displayed relation is checked directly. C↔F is shown separately as correspondence.

In [ ]:
sns.set_theme(style='white')
fig,axes=plt.subplots(5,3,figsize=(17,24))
for ax,row in zip(axes.flat,results.itertuples()):
    t=overlap.loc[overlap.coarse.eq(row.coarse)&overlap.fine.eq(row.fine)]
    values=t.pivot(index='coarse_cluster',columns='fine_cluster',values='pct_of_fine')
    sns.heatmap(values,annot=True,fmt='.0f',vmin=0,vmax=100,cmap='Blues',cbar=False,ax=ax)
    ax.set(title=f'{row.coarse} → {row.fine}: R={row.refinement_purity:.3f}',
           xlabel=f'{row.fine} cluster (column percentages)',ylabel=f'{row.coarse} cluster')
fig.tight_layout(); fig.savefig(OUTPUT/'all_pairwise_overlaps.png',dpi=140); plt.show()
strong=results.loc[results.coarse_k.lt(results.fine_k)&results.refinement_purity.ge(.90)&results.worst_fine_purity.ge(.80)]
display(Markdown('### Directly verified strong approximate-refinement relationships'))
display(strong.round(4))
positions={'A':(0,0),'B':(1,0),'C':(0,1),'F':(1,1),'D':(.5,2),'E':(.5,3)}
fig,ax=plt.subplots(figsize=(10,9))
for name,(x,y) in positions.items():
    encoding,balance,k=solutions[name]
    ax.text(x,y,f'{name}: {encoding}, k={k}\n{balance.replace("_"," ")}',ha='center',va='center',
            bbox=dict(boxstyle='round',facecolor='#e7eff8',edgecolor='#4672a3'),fontsize=11)
for row in strong.itertuples():
    x1,y1=positions[row.coarse];x2,y2=positions[row.fine]
    ax.annotate('',xy=(x2,y2-.12),xytext=(x1,y1+.12),arrowprops=dict(arrowstyle='->',color='#4672a3',connectionstyle='arc3,rad=.08'))
    ax.text((x1+x2)/2+.06,(y1+y2)/2,f'R {row.refinement_purity:.2f}\nmin {row.worst_fine_purity:.2f}',fontsize=9)
ax.set(xlim=(-.4,1.5),ylim=(-.4,3.4),title='Approximate refinement graph: directly checked relationships only')
ax.axis('off');fig.tight_layout();fig.savefig(OUTPUT/'approximate_refinement_graph.png',dpi=160);plt.show()


## Conclusions from the observed assignments

**There is no common refinement lattice among all six retained partitions.** Only D→E passes the illustrative joint screen R≥0.90 and minimum finer-cluster purity≥0.80. Thresholds are descriptive, not independent validation.

**D→E (0–1–3 worry-heavy k=7→8) is a convincing approximate refinement:** overall purity 97.8%, weakest finer-cluster purity 89.1%. The k=7 crime group splits roughly equally into crime-only and combined crime–traffic worry. The other groups are largely preserved; the drinking-water group has more boundary movement, with 85.5% remaining in its main successor.

**C→D (same encoding, k=6→7) is not simply one split:** purity is 82.8%, weakest finer group 52.6%. Similar profile names concealed redistribution, so descriptions alone should not be used to assert nested membership.

**F→D/E has high overall but imperfect local nesting:** 94.6%/95.2% overall purity, weakest group 72.6%/72.9%. The F water/broad-concern group divides between drinking-water emphasis and broadly high worry, but those groups also recruit respondents from other F groups. These are plausible approximate connections excluded by the stricter per-cluster graph criterion.

**A and B do not provide universal parent partitions:** A→B is 86.2% overall with a weakest fine-cluster purity of 46.9%; A→D is 77.6%; B→D is 63.2%. The balanced broad-harm group crosses worry-focused profiles rather than simply subdividing into them.

**C versus F is correspondence, not refinement:** optimal one-to-one weighted agreement is 83.2%, unweighted ARI 0.819; this leaves meaningful encoding sensitivity despite similar labels. No exact refinement relationship was found. Disconnected graph nodes are not invalid clusterings: the graph tests nesting, not quality.
